# RG3 전처리 및 가스12개/최고미성형 저장 모델 생성


In [ ]:
from pathlib import Path
import os,sys,json,joblib,nbformat
import numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from scipy.special import expit
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'rg3/01_preprocess_train.ipynb').exists())
WORK=ROOT/'rg3/internal';MODEL_DIR=ROOT/'rg3/models';MODEL_DIR.mkdir(exist_ok=True)
def source(name):
 return ''.join(c.source for c in nbformat.read(WORK/'notebooks'/name,4).cells if c.cell_type=='code')
gas_source=source('154_rg3_gas_model_export.ipynb')
gas_source=gas_source.replace('for seed in [42,7,101]:','for seed in [42]:').replace('for c in [.1,1.,10.]','for c in [1.]').replace("fit_conditional(d,y,'thermal',[42,7,101])","fit_conditional(d,y,'thermal',[42])").replace('for seeds in [[42,7,101],[11,23,37],[13,29,53]]:','for seeds in [[42]]:').replace('(vote>=2)','(vote>=1)')
exec(gas_source[:gas_source.index('root=Path.cwd()')])
def transform(imp,sc,x):
 return np.clip(sc.transform(imp.transform(x)),-12,12)
def fit_scale(x,normal):
 imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(x[normal]);sc=RobustScaler().fit(imp.transform(x)[normal])
 return imp,sc,transform(imp,sc,x)
def joint_features(x):
 j=np.maximum(0,np.column_stack([np.minimum(-x[:,0],x[:,2]),np.minimum(x[:,3],-x[:,4]),np.minimum(-x[:,5],x[:,6]),np.minimum(-x[:,0],-x[:,1]),np.minimum(x[:,7],-x[:,8]),np.minimum(x[:,9],x[:,10])]))
 return np.column_stack([j.max(axis=1),np.sort(j,axis=1)[:,-2],j.mean(axis=1),(j>1).sum(axis=1)])
def fit_mean_models(a,y,seeds,thermal):
 models=[]
 for cls in [RandomForestClassifier,ExtraTreesClassifier]:
  for seed in seeds:
   models.append((1/(5*len(seeds)),cls(n_estimators=100 if thermal else 120,max_depth=4,min_samples_leaf=2 if thermal else 3,class_weight='balanced',random_state=seed,n_jobs=2).fit(a,y)))
 for c in [.1,1.]:models.append((.2,SVC(C=c,class_weight='balanced',gamma='scale').fit(a,y)))
 models.append((.2,LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(a,y)))
 return models
def mean_scores(models,a):
 return sum(w*(expit(m.decision_function(a)) if isinstance(m,SVC) else m.predict_proba(a)[:,1]) for w,m in models)
def local_short(peer,x,context,timestamps):
 xx=peer['imp'].transform(x);cx=peer['sc'].transform(peer['ci'].transform(context))
 near=peer['nn'].kneighbors(cx,return_distance=False);out=[]
 for row,ids,ts in zip(xx,near,timestamps):
  ids=ids[peer['timestamps'][ids]!=np.datetime64(ts)][:30]
  if not len(ids):ids=np.arange(len(peer['x']))
  out.append((row-np.median(peer['x'][ids],axis=0))/np.maximum(peer['x'][ids].std(axis=0),peer['floor']))
 return np.asarray(out)
def predict_short(bundle,f):
 a=transform(bundle['A']['imp'],bundle['A']['sc'],f[bundle['columns']['std']].to_numpy(float))
 full=expit(bundle['A']['full'].decision_function(a))
 experts=np.max([mean_scores(ms,a) for ms in bundle['A']['modes']],axis=0)
 A=np.sqrt(full*experts)
 b=local_short(bundle['B']['peer'],f[bundle['columns']['physical']].to_numpy(float),f[bundle['columns']['context']].to_numpy(float),pd.to_datetime(f.TimeStamp).to_numpy())
 b=np.clip(bundle['B']['sc'].transform(b),-12,12)
 B=mean_scores(bundle['B']['models'],b)
 C=np.mean([mean_scores(c['models'],transform(c['imp'],c['sc'],f[bundle['columns']['shock']].to_numpy(float))) for c in bundle['C']],axis=0)
 dd=joint_features(transform(bundle['D']['imp'],bundle['D']['sc'],f[bundle['columns']['joint']].to_numpy(float)))
 D=bundle['D']['model'].predict_proba(dd)[:,1]
 cuts=bundle['cuts']
 flags=np.column_stack([A>=cuts['A'],B>=cuts['B'],(C>=cuts['C'])&(f.Barrel_Temperature_5__past10_std.to_numpy()>=cuts['barrel5']), (D>=cuts['D'])&(-f.Hopper_Temperature__dev10.to_numpy()>=cuts['hopper'])])
 return flags.any(axis=1),np.column_stack([A,B,C,D])



In [ ]:
import shutil,gzip
rawdir=WORK/'data/raw_input';rawdir.mkdir(parents=True,exist_ok=True)
for name in ['labeled_data.csv','moldset_labeled.csv']:
 shutil.copy2(ROOT/'dataset/original'/name,rawdir/name)
with gzip.open(ROOT/'dataset/original/unlabeled_data.csv.gz','rb') as src,(rawdir/'unlabeled_data.csv').open('wb') as dst:shutil.copyfileobj(src,dst)
for p in (ROOT/'dataset/kamp').glob('*.csv'):shutil.copy2(p,rawdir/p.name)
legacy_original=WORK/'data/original';legacy_original.mkdir(parents=True,exist_ok=True)
for p in rawdir.glob('*.csv'):shutil.copy2(p,legacy_original/p.name)
for p in (ROOT/'dataset/kamp').glob('*.csv'):shutil.copy2(p,WORK/'data'/p.name)
# 기존 전처리를 새 실행 루트에서 재구성. 모델/예측 CSV를 입력으로 쓰지 않는다.
previous=Path.cwd()
try:
 os.chdir(WORK)
 prep=source('158_rg3_prepare_from_local_raw.ipynb')
 # 새 helper .py를 작성하지 않고 이미 제공된 함수와 노트북 사용.
 exec(prep,{'__name__':'__main__'})
 # 공식 가스12개 OOF를 처음부터 재현해 경계값 산정.
 n=nbformat.read(WORK/'notebooks/193_rg3_fresh12_full_reproduction.ipynb',4)
 ns={'__name__':'__main__'}
 for c in n.cells:
  if c.cell_type=='code':exec(c.source,ns)
 # 최고 미성형의 구간 제외 개발평가 재현.
 for filename in ['133_rg3_shortshot_sparse_modes.ipynb','139_rg3_shortshot_thermal_seed_majority.ipynb','141_rg3_shortshot_joint_tail_confirmation.ipynb','143_rg3_shortshot_fixed_candidate_review.ipynb']:
  if not (WORK/'notebooks'/filename).exists():
   matching=list((WORK/'notebooks').glob(filename.split('_')[0]+'_*'));assert len(matching)==1;filename=matching[0].name
  print('Shortshot reproduction',filename,flush=True)
  temp={'__name__':'__main__'}
  for c in nbformat.read(WORK/'notebooks'/filename,4).cells:
   if c.cell_type=='code':exec(c.source,temp)
 # 미성형145의 학습 정의만 사용; 비라벨 대입/예측은 실행하지 않음.
 setup=source('145_rg3_shortshot_unlabeled_application.ipynb')
 shortns={'__name__':'__main__'};exec(setup[:setup.index('u_all=pd.read_csv')],shortns)
finally:os.chdir(previous)
d=pd.read_csv(WORK/'reports/tables/original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
gas=d[d['product'].eq('RG3')&(d.PassOrFail.eq(0)|d.Reason.eq('가스'))].reset_index(drop=True)
cal=pd.read_csv(WORK/'reports/tables/rg3_193_fresh12_predictions.csv',parse_dates=['TimeStamp']).set_index('TimeStamp').loc[gas.TimeStamp]
normal=gas.PassOrFail.eq(0).to_numpy()
cuts={k:float(np.quantile(cal.loc[normal,k+'_score'],q,method='higher')) for k,q in [('baseline',.99),('expert',.995),('thermal',.975)]}
# 압력 CSV column 이름은 pressure_score.
cuts['pressure_42_7_101']=float(np.quantile(cal.loc[normal,'pressure_score'],.99,method='higher'))
dd=gas.Barrel_Temperature_5__delta1.to_numpy();cuts['barrel5_delta']=float(np.quantile(dd[normal&np.isfinite(dd)],.95,method='higher'))
zcal=[]
for block in gas.comparison_block.unique():
 q=np.flatnonzero(gas.comparison_block.eq(block));ref=np.flatnonzero(~gas.comparison_block.eq(block))
 zcal.extend(zip(q,ns['local_residual'](ref,q,'thermal')[:,4]))
cuts['thermal_score']=cuts.pop('thermal');cuts['thermal_gate']=float(np.quantile(np.array([v for _,v in sorted(zcal)])[normal],.05))
bundle=fit_gas_bundle(gas,cuts)
assert len(bundle['pressure'])==1 and sum(len(g) for g in bundle['thermal']['trees'])+len(bundle['thermal']['linear'])==3
save_bundle(bundle,MODEL_DIR/'gas12.joblib')
na,nb,nd,nc=[shortns[k] for k in ['na','nb','nd','nc']]
lab=na['lab'];use=na['use'];yy=na['y'][use];norm=yy==0;ref=np.flatnonzero(use);g=na['g']
def short_scores(ref,q):
 return np.column_stack([na['predict'](ref,q)['std24_geometric'],nb['predict'](ref,q)['localphysical12_mean'],np.mean([v['predict'](ref,q)['shock6_mean'] for v in nc],axis=0),nd['predict'](ref,q)])
inner=np.full((len(ref),4),np.nan)
for block in pd.unique(g[ref]):
 mask=g[ref]==block;inner[mask]=short_scores(ref[~mask],ref[mask])
cuts_s={'A':float(na['threshold'](yy,inner[:,0],.03)),'B':float(nb['threshold'](yy,inner[:,1],.03)),'C':float(np.quantile(inner[norm,2],.99,method='higher')),'D':float(np.quantile(inner[norm,3],.995,method='higher'))}
dense=nc[0]['dense']
for key,col,sign in [('barrel5','Barrel_Temperature_5__past10_std',1),('hopper','Hopper_Temperature__dev10',-1)]:
 a=sign*dense[col].to_numpy()[ref];cuts_s[key]=float(np.quantile(a[norm&np.isfinite(a)],.95,method='higher'))
imp,sc,a=fit_scale(na['views']['std24'][ref],norm)
full=SVC(C=1.,class_weight='balanced',gamma='scale').fit(a,yy)
pos=np.flatnonzero(yy==1);neg=np.flatnonzero(norm)
modes=KMeans(n_clusters=2,n_init=10,random_state=42).fit_predict(a[pos])
mode_models=[]
for mode in np.unique(modes):
 ids=np.r_[neg,pos[modes==mode]];models=[]
 for C in [.1,1.]:models.append((.25,SVC(C=C,class_weight='balanced',gamma='scale').fit(a[ids],yy[ids])))
 for C in [.1,1.]:models.append((.25,LogisticRegression(C=C,class_weight='balanced',max_iter=3000).fit(a[ids],yy[ids])))
 mode_models.append(models)
A={'imp':imp,'sc':sc,'full':full,'modes':mode_models}
x=nb['views']['physical12'][ref];context=nb['context'][ref]
imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(x[norm]);ci=SimpleImputer(strategy='median').fit(context[norm]);cx=ci.transform(context[norm]);cs=RobustScaler().fit(cx)
peer={'imp':imp,'ci':ci,'sc':cs,'x':imp.transform(x[norm]),'timestamps':lab.TimeStamp.to_numpy()[ref][norm]}
peer['floor']=np.maximum(peer['x'].std(axis=0)*.2,1e-6);peer['nn']=NearestNeighbors(n_neighbors=min(31,norm.sum())).fit(cs.transform(cx))
localized=local_short(peer,x,context,lab.TimeStamp.to_numpy()[ref]);bs=RobustScaler().fit(localized[norm])
B={'peer':peer,'sc':bs,'models':fit_mean_models(np.clip(bs.transform(localized),-12,12),yy,[42,7],False)}
C=[]
for seeds in [[42,7],[11,23],[101,37]]:
 imp,sc,a=fit_scale(nc[0]['thermal']['shock6'][ref],norm);C.append({'imp':imp,'sc':sc,'models':fit_mean_models(a,yy,seeds,True)})
imp,sc,a=fit_scale(nd['jointbase'][ref],norm)
D={'imp':imp,'sc':sc,'model':LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(joint_features(a),yy)}
f=d[d['product'].eq('RG3')].set_index('TimeStamp').loc[lab.TimeStamp].reset_index()
columns={}
for key,array in [('std',na['views']['std24']),('physical',nb['views']['physical12']),('shock',nc[0]['thermal']['shock6']),('joint',nd['jointbase']),('context',nb['context'])]:
 columns[key]=[key+'_'+str(i) for i in range(array.shape[1])]
 for j,col in enumerate(columns[key]):f[col]=array[:,j]
smodel={'A':A,'B':B,'C':C,'D':D,'cuts':cuts_s,'columns':columns,'sensor_names':shortns['sensor_names'],'training_normal':566,'training_shortshot':8}
p,s=predict_short(smodel,f)
# 학습점수 저장 재현 검사만 위해 원본 predict의 구간 assertion을 해제한다. OOF 검사는 이미 원본 assertion 유지 상태로 실행했다.
import ast
tree=ast.parse(setup[:setup.index('u_all=pd.read_csv')])
definitions=[node.value.args[0].value for node in tree.body if isinstance(node,ast.Expr) and isinstance(node.value,ast.Call) and isinstance(node.value.func,ast.Name) and node.value.func.id=='exec' and isinstance(node.value.args[0],ast.Constant)]
for target,definition in list(zip([na,nb,nd],definitions))+[(v,shortns['sourceC'].replace('for seed in [42,7]:','for seed in '+str(seeds)+':')) for v,seeds in zip(nc,[[42,7],[11,23],[101,37]])]:
 body=definition[definition.index('def predict('):]
 if 'def threshold(' in body:body=body[:body.index('def threshold(')]
 exec(body.replace('assert set(g[ref]).isdisjoint(g[q])','pass # fullfit score reconstruction only'),target)
expected=short_scores(ref,np.arange(len(lab)))
print('COMPONENT MAXDIFF',np.max(abs(s-expected),axis=0),flush=True)
assert np.allclose(s,expected,atol=1e-10),np.max(abs(s-expected))
joblib.dump(smodel,MODEL_DIR/'shortshot_best.joblib',compress=3)
processed=ROOT/'dataset/processed/rg3';processed.mkdir(parents=True,exist_ok=True)
f.to_csv(processed/'rg3_input_features.csv',index=False,encoding='utf-8-sig')
gas.to_csv(processed/'rg3_gas_training.csv',index=False,encoding='utf-8-sig')
f[use].to_csv(processed/'rg3_shortshot_training.csv',index=False,encoding='utf-8-sig')
print('Models saved; saved shortshot inference scores match original fullfit',len(f))

